# MethylSeg Figure Workflow

This notebook prepares representative WGBS, HM450K, and TCGA inputs and runs the same MethylSeg figure-generation steps for each case.

In [ ]:
%load_ext autoreload
%autoreload 2

In [ ]:
from pathlib import Path

from methylseg import MethylDataPrep, MethylSegPathway
from utils.figures_utils import RESULTS_DIR, load_sample

In [ ]:
OUT_DIR = Path("out") / "methyl_seg_figures"
OUT_DIR.mkdir(exist_ok=True, parents=True)

DISPLAY_CHROM = "chr1"
COLON_REGION = {
    "region_chrom": "chr1",
    "region_start": 190_000_000,
    "region_end": 195_000_000,
}
TCGA_REGION = {
    "region_chrom": "chr1",
    "region_start": 2_000_000,
    "region_end": 4_000_000,
}

In [ ]:
def prepare_sample(
    *,
    meth_file,
    sample_id,
    resolution,
    min_coverage=None,
):
    prep_kwargs = {
        "meth_file": meth_file,
        "sample_id": sample_id,
        "resolution": resolution,
        "remove_low_coverage_like_cpgs": True,
    }
    if min_coverage is not None:
        prep_kwargs["min_coverage"] = min_coverage

    sample_info, sample_info_removed = MethylDataPrep(**prep_kwargs).prepare()
    print(f"Prepared {sample_info.sample_id} ({resolution})")
    return sample_info, sample_info_removed


def write_tcga_sample(sample_id: str) -> Path:
    tcga_path = OUT_DIR / "inputs" / f"{sample_id}.tsv"
    tcga_path.parent.mkdir(exist_ok=True, parents=True)
    load_sample(sample_id).to_csv(tcga_path, index=False, header=False, sep="\t")
    print(f"Wrote {tcga_path}")
    return tcga_path


def get_plot_region_kwargs(sample_id: str) -> dict:
    plot_kwargs = {"chrom": DISPLAY_CHROM}
    if sample_id.lower().startswith("tcga"):
        plot_kwargs.update(TCGA_REGION)
    else:
        plot_kwargs.update(COLON_REGION)
    return plot_kwargs


def plot_case(model, sample_info, sample_info_removed,):
    plot_region_kwargs = get_plot_region_kwargs(sample_info.sample_id)

    model.analyzer.plot_feature_distributions_by_kmeans_state()
    model.plot_labels(
        label_source="kmeans",
        sample_info=sample_info,
        sample_info_removed=sample_info_removed,
        label_title="KMeans state",
        **plot_region_kwargs,
    )
    model.plot_labels(
        label_source="hmm",
        sample_info=sample_info,
        sample_info_removed=sample_info_removed,
        label_title="HMM state",
        **plot_region_kwargs,
    )
    model.plot_embedding(
        label_source="kmeans",
        sample_info=sample_info,
        chrom=plot_region_kwargs["chrom"],
        method="pca",
        include_biplot=True,
        include_metrics=False,
        label_title="KMeans state",
        hexbin=True,
    )


def run_case(
    *,
    label,
    meth_file,
    sample_id,
    resolution,
    out_subdir,
    min_coverage=None,
):
    print(f"=== {label} ===")
    sample_info, sample_info_removed = prepare_sample(
        meth_file=meth_file,
        sample_id=sample_id,
        resolution=resolution,
        min_coverage=min_coverage,
    )
    model = MethylSegPathway(
        out_dir=OUT_DIR / out_subdir,
        train_sample_info=sample_info,
    )
    model.run_pathway()
    plot_case(
        model,
        sample_info,
        sample_info_removed,
    )
    return {
        "sample_info": sample_info,
        "sample_info_removed": sample_info_removed,
        "model": model,
    }

## WGBS Example

In [ ]:
wgbs_results = run_case(
    label="WGBS colon-primary-tumor_1",
    meth_file=RESULTS_DIR / "01_region_calling_analysis/methylseg/WGBS_colon-primary-tumor_1_meth/prep/wgbs.beta",
    sample_id="colon-primary-tumor_1",
    resolution="wgbs",
    out_subdir="wgbs",
    min_coverage=10,
)

## HM450K Example

In [ ]:
hm450k_results = run_case(
    label="HM450K colon-primary-tumor_1",
    meth_file=RESULTS_DIR / "01_region_calling_analysis/methylseg/WGBS_colon-primary-tumor_1_meth/prep/450k.beta",
    sample_id="colon-primary-tumor_1",
    resolution="450k",
    out_subdir="hm450k",
    min_coverage=10,
)

## TCGA HM450K Example

In [ ]:
tcga_sample_id = "TCGA-BD-A3EP-01A"
tcga_meth_file = write_tcga_sample(tcga_sample_id)

tcga_results = run_case(
    label=f"TCGA HM450K {tcga_sample_id}",
    meth_file=tcga_meth_file,
    sample_id=tcga_sample_id,
    resolution="450k",
    out_subdir="tcga_hm450k",
)